# PPRS: Week 11 Part 2
# Spatial data analysis and batch processing in Python

In [ ]:
# run this to display graphs nicely in this notebook

%matplotlib inline


## Calibrating the biomass model

Now you have a 10 m resolution CHM. You also have some field data in a csv file stored in:

    data/ground/ground_data.csv
    
You can calibrate the model to predict biomass from lidar height. The ground data can be read in using the ***numpy.loadtxt()*** function to store the biomass (AGBD) and coordinates (x and y) in RAM.

In [ ]:
import numpy as np
import rasterio

groundname='data/ground/ground_data.csv'

# load data in to RAM
agbd,x,y=np.loadtxt(groundname, usecols=(1,2,3), unpack=True, dtype=float,skiprows=1,delimiter=',')

# Here agbd is in kg/ha. Let's use the more common Mg/ha
agbd=agbd/1000

print(agbd,'Mg/ha')


Now we want to intersect this with our CHM raster to create an array with the biomass for each plot along with the mean CHM. We can then plot that data to see what type of mathematical model would be appropriate to predict biomass from lidar data.

The script below will read in the raster layer and then return an array of CHM values corresponding to the coordinates you read in from the field data file. In this code we are finding a the nearest grid point in the CHM for each ground data point, and using the value in that pixel.

**Add some code to make a scatterplot of biomass against mean canopy height. What type of relationship do you see?**

In [ ]:
# read the CHM geotiff in to RAM
chmName='data/ALS/merged_CHM_10m.tif'
chmFile=rasterio.open(chmName)
chmData=chmFile.read(1)

# find the top left corner coordinate
xOrigin=chmFile.transform[2]
yOrigin=chmFile.transform[5]

# get a list of pixel coordinates for the ground plots (there are other packages for intersecting raster and vector data)
xInds=np.array((x-xOrigin)/chmFile.res[0],dtype=int)
yInds=np.array((yOrigin-y)/chmFile.res[1],dtype=int)

# extract the values to make a new array of mean canopy height
meanCH=chmData[yInds,xInds]

# add some code to make a scatterplot of biomass (agbd) against mean canopy height (meanCH)


Now you have viewed the data and chosen an appopriate mathamatical model (was it linear, logarithmic etc.?), if it is linear you can perform linear regression to get the equation for your biomass model. 

The code below shows you can example of fitting a line of best fit to two random datasets, and calculates the correlation (Pearson's $r$ value). Pearson's $r$ is the correlation between $x$ and $y$, which is also returned by `linregress`. The difference is that `pearsonr` returns its value *and the statistical significance*. $R^2$ is the square of Pearson's $r$.

**Modify that code to find the equation needed to predict biomass from mean canopy height and report the strength of the correlation.**

In [ ]:

# packages to do linear regression and Pearson's correlation
from scipy.stats import linregress, pearsonr

# make some fake data
x=np.random.random((20))*100
y=x*3+20+40*(np.random.random((20))-0.5)

# fit a line of best fit. See manual for scipy.stats.linregress for more details
m, c, r, _, _ = linregress(x,y)

print("Line of best fit parameters are m",round(m,3),"c",round(c,3))

# find the correlation. See manual for scipy.stats.pearsonr
print('Correlation is ',pearsonr(x,y))

# the first number returned by pearsonr is the correlation, 99.28% for this fake data


# repeat with the real data



## Making the biomass map


Now you have the parameters needed to parameterise a biomass model. **In the cell below, write some code to read in the CHM again (you can reuse the code above) and make a new array of biomass using the model you have just calibrated.** If this was a linear model, the biomass will be:

$biomass = m \times chmData + c $

You can use ***plt.imshow()*** to print it as an image on your screen, or modify the CHM writing function above to write the biomass raster layer as a new geotiff. Use whichever approach you prefer.

In [ ]:
# read the CHM geotiff in to RAM

# filter missing data
chmData[chmData<0.0]=np.nan

# apply the model

# print to screen 


**Using numpy operations, what is the mean biomass for this valley? What is the maximum biomass for any 10 m pixel?**

In [ ]:
# filter out missing data
usableBiomass=biomass[np.isnan(biomass)==False]


# find the mean and max


## Writing the biomass map to file for reuse

We have displayed the biomass map to screen above. Often we will want to save it for later analysis, either in python or in GIS. We can do this using rasterio with the example below. We can now open this geotiff in GIS or other software.

In [ ]:

# set a filename
outName='biomass.tif'

# create a new raster file, taking the geolocation information from the CHM file
newfile=rasterio.open(outName,mode="w",driver="GTiff",height=chmFile.height,width=chmFile.width,\
                      count=1,dtype=biomass.dtype,crs=chmFile.crs,chmFile=chmFile.transform)
newfile.write(biomass, 1)

print('Written to',outName)


## A comment on automation


As you have seen above, making a biomass map with Python requires quite a few functions, some using some unintuitive aspects of the language. However, you now have a workflow to process a set of geotiff tiles through to biomass. You can rerun the whole workflow by pressing the double arrow button at the top of the screen.

You can do this for any data by downloading new tiles and putting them in the directories above. You can make adjustments to the workflow instructions (eg. change the resolution you do the calculation at, or add in an extra bit of processing) and then reurn everything with a single button.

## Visually assessing the results

You have now produced a biomass map entirely in Python. Whilst python does allow you to make plots of maps and to calculate summary statistics, it can be clunky to assess one map against other. For that task, GIS allows you to more interactively work with the data.

Download the biomass map you have made (go to the tab to the left of this, navigate to the map and tick and download it). Load this in to QGIS (either on your own computer or Apps.ed) and overlay it with a Bing aerial image.

Can you see any obvious errors in the biomass map? Can you see a way to limit your analysis to a particular estate or a particular land type?